# tAIlor — Qwen2.5-VL wardrobe scanner (Colab)

Loads **Qwen/Qwen2.5-VL-3B-Instruct** and serves:
- `GET /health`
- `POST /v1/chat/completions` (stylist + text)
- `POST /v1/wardrobe/scan` (image → wardrobe JSON)

Qwen weights download **without** an HF token. Runtime → Change runtime type → **T4 GPU**.

### Step 1: Confirm GPU allocation

In [ ]:
!nvidia-smi

### Step 2: Install dependencies

In [ ]:
!pip install -q "torch" "transformers>=4.49.0" "accelerate" "fastapi" "uvicorn" "pillow" "uvicorn[standard]"

### Step 3: Fetch `colab_server.py` from the repo
Or upload the file from your local checkout via the Colab file browser.

In [ ]:
# Prefer the PR / branch raw URL while testing; switch to main after merge.
!wget -q -O /content/colab_server.py https://raw.githubusercontent.com/areebasa02/tAIlor/cursor/wardrobe-qwen-vl-378f/colab_server.py
!wc -l /content/colab_server.py

### Step 4: Start the VL server
No HF token required for Qwen2.5-VL.

In [ ]:
import os
os.environ["LLM_MODEL"] = "Qwen/Qwen2.5-VL-3B-Instruct"
# Optional only if your environment blocks anonymous downloads:
# os.environ["HF_TOKEN"] = "hf_..."
!nohup python /content/colab_server.py > /content/stylist-server.log 2>&1 &
print("Server starting — first load downloads ~7GB weights; wait for health ok.")

### Step 5: Wait until `/health` returns status ok

In [ ]:
import time, requests
for i in range(90):
    try:
        payload = requests.get("http://127.0.0.1:8000/health", timeout=3).json()
        print(payload)
        if payload.get("status") == "ok":
            break
    except Exception as exc:
        print(f"waiting… ({i}) {exc}")
        time.sleep(5)
else:
    print(open("/content/stylist-server.log", encoding="utf-8").read()[-2000:])

### Step 6: Start Cloudflare tunnel

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
!nohup /content/cloudflared tunnel --url http://127.0.0.1:8000 --no-autoupdate > /content/cloudflared.log 2>&1 &

### Step 7: Print the public tunnel URL

In [ ]:
import re, time
for _ in range(30):
    log = open("/content/cloudflared.log", encoding="utf-8").read()
    match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", log)
    if match:
        print(match.group(0))
        print("Set local .env:")
        print(f"LLM_BASE_URL={match.group(0)}/v1")
        print("LLM_MODEL=Qwen/Qwen2.5-VL-3B-Instruct")
        print("WARDROBE_SCAN_MOCK=false")
        break
    time.sleep(2)

### Optional: smoke-test wardrobe scan with a tiny PNG

In [ ]:
import base64, io, json, requests
from PIL import Image

img = Image.new("RGB", (64, 64), (0, 120, 128))
buf = io.BytesIO()
img.save(buf, format="PNG")
payload = {
    "image": base64.b64encode(buf.getvalue()).decode("ascii"),
    "mimeType": "image/png",
}
r = requests.post("http://127.0.0.1:8000/v1/wardrobe/scan", json=payload, timeout=120)
print(r.status_code)
print(json.dumps(r.json(), indent=2)[:1200])